# Interspeech 2026 — Voice Design Consistency via Continuation


## Imports

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers.utils.notebook import NotebookProgressBar

import torchaudio.transforms as T

from voicestudio.utils.audio_utils import show_waveform
import matplotlib.pyplot as plt

In [3]:
import transformers
transformers.logging.set_verbosity_error()

### Check GPU Availability

In [4]:
!nvidia-smi

Sat Jul 11 06:28:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 5000 Blac...    Off |   00000000:01:00.0 Off |                  Off |
| 30%   26C    P8             11W /  300W |      18MiB /  48935MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Set CUDA Device Number
DEVICE_NUM = 0

if torch.cuda.is_available():
    device = torch.device(f"cuda:{DEVICE_NUM}")
else:
    device = torch.device("cpu")
    DEVICE_NUM = -1

device_map = f"cuda:{DEVICE_NUM}" if DEVICE_NUM >= 0 else "cpu"
print(f"INFO: Using device - {device}")

INFO: Using device - cuda:0


## Datasets

In [6]:
from spk_incon.datasets import LIBRITTS_P_Custom
from spk_incon.datasets.libritts_p3 import download_libritts_p_metadata

In [7]:
from spk_incon.metrics.presets import DatasetType, GenerationMethod, SynthesisConfig, ModelType
from spk_incon.metrics.strategies import create_strategy
from spk_incon.datasets import DatasetType, create_dataset

from spk_incon.utils.evaluate import EvaluationPipeline

In [8]:
DATA_ROOT = "./data"
Z_THRESHOLD = 2
MIN_GRP_SIZE = 0
URL = "https://dolab-data.duckdns.org/api/public/dl/-qA96ilN"

In [9]:
if not os.path.isfile(os.path.join(DATA_ROOT, "train-clean-100.tar.gz")):
    !wget -O "./data/train-clean-100.tar.gz" {URL}

In [10]:
download_libritts_p_metadata(root=DATA_ROOT, annotator="df1")
curated_dataset = LIBRITTS_P_Custom(
    root=DATA_ROOT, download=True, max_z_score=Z_THRESHOLD, min_group_size=MIN_GRP_SIZE
)

[INFO] Loading cached dataset from data/.cache/libritts_p_train-clean-100/dataset...


[INFO] Filtering outliers (max_z_score=2)...


[INFO] Filtered: 33187 -> 29679 samples.
[INFO] Filtering groups with fewer than 0 samples...


[INFO] Filtered: 29679 -> 29679 samples.


In [11]:
test_config = SynthesisConfig()
test_dataset_type = DatasetType.LIBRITTS
test_dataset_config = test_config.get_dataset_config(test_dataset_type.value)

test_dataset = create_dataset(test_dataset_type, test_dataset_config, root_dir="./data")

INFO: Loading 'test.other' split of LibriTTS dataset...


Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/116 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/63 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/116 [00:00<?, ?it/s]

Loaded LibriTTS 'test.other' split with 4705 samples


## Models

In [12]:
from transformers import AutoTokenizer, AutoProcessor

from voicestudio.models.parler_tts import ParlerTTSForConditionalGeneration
from voicestudio.models.qwen3_tts import Qwen3TTSForConditionalGeneration

### Model Selection

In [13]:
# Model select
#model_id = "parler-tts/parler-tts-mini-v1"
#model_id = "parler-tts/parler-tts-large-v1"
#model_id = "parler-tts/parler-tts-mini-v1.1"

#model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"

In [14]:
# Model loading
if "parler" in model_id.lower():
    model = ParlerTTSForConditionalGeneration.from_pretrained(
        model_id, device_map=device_map
    )
    config = model.config
    model_dtype = model.dtype
    processor = AutoProcessor.from_pretrained(model_id)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
elif "qwen" in model_id.lower():
    model = Qwen3TTSForConditionalGeneration.from_pretrained(
        model_id, device_map=device_map, dtype=torch.bfloat16, attn_implementation="flash_attention_2",
    )
    config = model.config
    model_dtype = model.dtype
    processor = AutoProcessor.from_pretrained(model_id, device_map=device_map)
    tokenizer = processor.tokenizer
else:
    pass

model.eval()


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen3TTSForConditionalGeneration(
  (talker): Qwen3TTSTalkerForConditionalGeneration(
    (model): Qwen3TTSTalkerModel(
      (layers): ModuleList(
        (0-27): 28 x Qwen3TTSTalkerDecoderLayer(
          (self_attn): Qwen3TTSTalkerAttention(
            (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
            (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
            (q_norm): Qwen3TTSRMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3TTSRMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3TTSTalkerTextMLP(
            (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
            (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
            (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
            (act_fn): SiLUActivation

## Single-Pass Continuation (2D causal, voice prompt visible)

Sequence: `[voice prompt] [role] [codec prefix] [fixed content + ref audio] [content] [gen audio]`

One contiguous causal forward with a standard 2D all-ones mask — voice prompt is kept at the front, so **both the ref audio and the continuation attend to it** (no 4D blocking).

In [15]:
import librosa

FIXED_CONTENT = "Hello, this is a fixed sentence used as the acoustic anchor for voice design retrieval."
STAGE2_BATCH = 25          # utterances per batched talker.generate call (memory hedge)
_REF_CACHE: dict = {}      # voice_prompt -> dict(ref_codes, voice_prompt)  (cached mode only)

_TALKER_GEN_KWARGS = dict(
    max_new_tokens=2048, min_new_tokens=2,
    do_sample=True, top_k=50, top_p=1.0, temperature=0.9,
    subtalker_dosample=True, subtalker_top_k=50, subtalker_top_p=1.0, subtalker_temperature=0.9,
    repetition_penalty=1.05,
    output_hidden_states=True, return_dict_in_generate=True,
)

# ---------------------------------------------------------------- Stage 1 (batched)
@torch.no_grad()
def _gen_refs(voice_prompts, gen_kwargs=None):
    """voice_prompts(중복 허용) 각각에 대해 ref codes 를 배치 생성해 리스트로 반환."""
    clean = {k: v for k, v in (gen_kwargs or {}).items() if k != "pad_token_id"}
    refs = []
    for i in range(0, len(voice_prompts), STAGE2_BATCH):
        chunk = voice_prompts[i:i + STAGE2_BATCH]
        inputs = processor.encode_voice_design(text=[FIXED_CONTENT] * len(chunk), instruct=chunk)
        out = model.generate(**inputs, **clean)
        for vp, codes in zip(chunk, out.audio_codes):
            refs.append(dict(ref_codes=codes.detach().to(device), voice_prompt=vp))
    return refs

@torch.no_grad()
def ensure_refs(voice_prompts, gen_kwargs=None):
    """CACHED: 캐시에 없는 unique voice prompt 만 생성해 _REF_CACHE 채움."""
    need = [vp for vp in dict.fromkeys(voice_prompts) if vp not in _REF_CACHE]
    for vp, ref in zip(need, _gen_refs(need, gen_kwargs)):
        _REF_CACHE[vp] = ref

@torch.no_grad()
def make_refs_fresh(voice_prompts, gen_kwargs=None):
    """NO-CACHE: 발화마다 fresh ref (중복 voice prompt 도 각각 새로 생성)."""
    return _gen_refs(voice_prompts, gen_kwargs)

# ---------------------------------------------------------------- Stage 2 helpers
def _common_talker_embeds(language="Auto"):
    cfg = model.config; talker = model.talker; long_dtype = torch.long
    tts_bos_embed, tts_eos_embed, tts_pad_embed = talker.text_projection(
        talker.get_text_embeddings()(torch.tensor(
            [[cfg.tts_bos_token_id, cfg.tts_eos_token_id, cfg.tts_pad_token_id]],
            device=device, dtype=long_dtype))
    ).chunk(3, dim=1)
    language_id = None if language.lower() == "auto" else cfg.talker_config.codec_language_id[language.lower()]
    if language_id is None:
        prefill = [[cfg.talker_config.codec_nothink_id, cfg.talker_config.codec_think_bos_id,
                    cfg.talker_config.codec_think_eos_id]]
    else:
        prefill = [[cfg.talker_config.codec_think_id, cfg.talker_config.codec_think_bos_id,
                    language_id, cfg.talker_config.codec_think_eos_id]]
    codec_emb_0 = talker.get_input_embeddings()(torch.tensor(prefill, device=device, dtype=long_dtype))
    codec_emb_1 = talker.get_input_embeddings()(torch.tensor(
        [[cfg.talker_config.codec_pad_id, cfg.talker_config.codec_bos_id]], device=device, dtype=long_dtype))
    return tts_bos_embed, tts_eos_embed, tts_pad_embed, torch.cat([codec_emb_0, codec_emb_1], dim=1)

def _build_stage2_item(target_text, ref, common, non_streaming_mode=False):
    talker = model.talker
    tts_bos_embed, tts_eos_embed, tts_pad_embed, codec_input_emb = common
    ref_codes = ref["ref_codes"]
    instruct_id = tokenizer(processor._build_instruct_text(ref["voice_prompt"]), return_tensors="pt").input_ids.to(device)
    if instruct_id.dim() == 1: instruct_id = instruct_id.unsqueeze(0)
    instruct_emb = talker.text_projection(talker.get_text_embeddings()(instruct_id))

    input_id = tokenizer(processor._build_assistant_text(target_text), return_tensors="pt").input_ids.to(device)
    if input_id.dim() == 1: input_id = input_id.unsqueeze(0)
    ref_id = tokenizer(processor._build_ref_text(ref.get("ref_text", FIXED_CONTENT)), return_tensors="pt").input_ids.to(device)
    if ref_id.dim() == 1: ref_id = ref_id.unsqueeze(0)

    role_emb = talker.text_projection(talker.get_text_embeddings()(input_id[:, :3]))
    pad_prefix = torch.cat(
        (tts_pad_embed.expand(-1, codec_input_emb.shape[1]-2, -1), tts_bos_embed), dim=1
    ) + codec_input_emb[:, :-1]
    prefix_embed = torch.cat((role_emb, pad_prefix), dim=1)

    icl_input_embed, trailing_text_hidden = model.generate_icl_prompt(
        text_id=input_id[:, 3:-5], ref_id=ref_id[:, 3:-2], ref_code=ref_codes,
        tts_pad_embed=tts_pad_embed, tts_eos_embed=tts_eos_embed, non_streaming_mode=non_streaming_mode,
    )
    talker_input_embed = torch.cat([instruct_emb, prefix_embed, icl_input_embed], dim=1)
    return talker_input_embed, trailing_text_hidden, ref_codes

# ---------------------------------------------------------------- Stage 2 (batched)
@torch.no_grad()
def stage2_continuation_batch(texts, refs, gen_kwargs=None, language="Auto", non_streaming_mode=False):
    cfg = model.config; talker = model.talker; long_dtype = torch.long
    common = _common_talker_embeds(language)
    tts_pad_embed = common[2]

    input_embeds_list, trailing_list, ref_codes_list = [], [], []
    for t, r in zip(texts, refs):
        emb, trail, rc = _build_stage2_item(t, r, common, non_streaming_mode)
        input_embeds_list.append(emb); trailing_list.append(trail); ref_codes_list.append(rc)

    original_lengths = torch.tensor([t.shape[1] for t in input_embeds_list])
    sequences_reversed = [t.squeeze(0).flip(dims=[0]) for t in input_embeds_list]
    padded_reversed = torch.nn.utils.rnn.pad_sequence(sequences_reversed, batch_first=True, padding_value=0.0)
    talker_input_embeds = padded_reversed.flip(dims=[1])
    B, max_len = talker_input_embeds.shape[0], talker_input_embeds.shape[1]
    indices = torch.arange(max_len).expand(B, -1)
    num_pads = max_len - original_lengths
    talker_attention_mask = (indices >= num_pads.unsqueeze(1)).long().to(device)

    pad_vec = tts_pad_embed.squeeze()
    seqs = [t.squeeze(0) for t in trailing_list]
    tl = [s.shape[0] for s in seqs]
    padded_hiddens = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0.0)
    ar = torch.arange(max(tl), device=padded_hiddens.device).expand(len(tl), -1)
    lt = torch.tensor(tl, device=padded_hiddens.device).unsqueeze(1)
    padded_hiddens[ar >= lt] = pad_vec
    trailing_text_hiddens = padded_hiddens

    talker_kwargs = dict(_TALKER_GEN_KWARGS)
    talker_kwargs["eos_token_id"] = cfg.talker_config.codec_eos_token_id
    talker_kwargs["suppress_tokens"] = [
        i for i in range(cfg.talker_config.vocab_size - 1024, cfg.talker_config.vocab_size)
        if i != cfg.talker_config.codec_eos_token_id
    ]
    if gen_kwargs:
        talker_kwargs.update({k: v for k, v in gen_kwargs.items() if k != "pad_token_id"})

    result = talker.generate(
        inputs_embeds=talker_input_embeds, attention_mask=talker_attention_mask,
        trailing_text_hidden=trailing_text_hiddens, tts_pad_embed=tts_pad_embed, **talker_kwargs,
    )

    talker_codes = torch.stack([hid[-1] for hid in result.hidden_states if hid[-1] is not None], dim=1)
    first_book = talker_codes[:, :, 0]
    is_stop = (first_book == cfg.talker_config.codec_eos_token_id)
    has_stop = is_stop.any(dim=1)
    stop_idx = torch.argmax(is_stop.int(), dim=1)
    eff_len = torch.where(has_stop, stop_idx, talker_codes.shape[1])

    outs = []
    for i in range(B):
        codes = talker_codes[i, :int(eff_len[i])]
        rc = ref_codes_list[i]
        outs.append(dict(audio_codes=[torch.cat([rc, codes], dim=0)], ref_code_lengths=[int(rc.shape[0])]))
    return outs

import random as _random
import numpy as _np
@torch.no_grad()
def make_refs_seedlocked(voice_prompts, gen_kwargs=None, seed=42):
    """NO-CACHE structure, but the RNG seed is reset right before EACH stage-1 ref
    generation. Because (voice_prompt + FIXED_CONTENT + seed) is then deterministic,
    every utterance sharing a voice prompt gets an *identical* ref anchor.
    (Same-vp refs are byte-identical, so we compute one per unique vp and map.)"""
    cache = {}
    for vp in dict.fromkeys(voice_prompts):
        _random.seed(seed); _np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        cache[vp] = _gen_refs([vp], gen_kwargs)[0]
    return [cache[vp] for vp in voice_prompts]


# ================= multi-reference anchors =================
@torch.no_grad()
def make_refs_multi(voice_prompts, gen_kwargs=None, seed=42, M=1, content=None):
    content = content if content is not None else FIXED_CONTENT
    clean = {k: v for k, v in (gen_kwargs or {}).items() if k != "pad_token_id"}
    cache = {}
    for vp in dict.fromkeys(voice_prompts):
        codes_list = []
        for j in range(M):
            random.seed(seed + j); np.random.seed(seed + j); torch.manual_seed(seed + j)
            if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed + j)
            inputs = processor.encode_voice_design(text=content, instruct=vp)
            out = model.generate(**inputs, **clean)          # normal sampling
            codes_list.append(out.audio_codes[0].detach().to(device))
        ref_codes = torch.cat(codes_list, dim=0)             # concat M anchors
        ref_text = " ".join([content] * M)                   # matches concatenated audio
        cache[vp] = dict(ref_codes=ref_codes, voice_prompt=vp, ref_text=ref_text)
    return [cache[vp] for vp in voice_prompts]

def continuation_decode(out):
    wavs, sr = processor.decode(out)
    cut = out["ref_code_lengths"][0] * processor.get_decode_upsample_rate()
    wavs = [w[cut:] for w in wavs]
    return wavs, sr


In [16]:
from pathlib import Path
import random

import numpy as np
import torch

import soundfile as sf


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class TestModel:
    @classmethod
    def seed_everything(cls, seed: int = 42):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    @classmethod
    def synthesize(
        cls,
        text: str,
        output_path: Path,
        reference_audio: Path | None = None,
        style_prompt: str | None = None,
        speaker_id: str | None = None
    ) -> bool:
        is_batched = isinstance(text, (tuple, list)) and len(text) > 1

        rng_state = {
            'random': random.getstate(),
            'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        cls.seed_everything(42)

        (output_path[0] if is_batched else output_path).parent.mkdir(parents=True, exist_ok=True)

        # Setup generation config
        generation_config = dict(
            pad_token_id=tokenizer.eos_token_id
        )

        # Normalize to lists
        texts = list(text) if is_batched else [text]
        prompts = (list(style_prompt) if isinstance(style_prompt, (list, tuple))
                   else [style_prompt] * len(texts))
        paths = list(output_path) if is_batched else [output_path]

        audio_values = []
        sample_rate = None

        if "parler" in model_id.lower():
            # Parler is not the target of this experiment — keep original behaviour
            inputs = dict(
                input_ids=tokenizer(style_prompt, return_tensors="pt").input_ids.to(device),
                prompt_input_ids=tokenizer(text, return_tensors="pt").input_ids.to(device)
            )
            outputs = model.generate(**inputs, **generation_config)
            audio_values.append(outputs.cpu().numpy().squeeze())
            sample_rate = config.audio_encoder.sampling_rate

        elif "qwen" in model_id.lower():
            # BATCHED continuation across utterances (GPU efficiency).
            vps = [p or FIXED_CONTENT for p in prompts]
            refs = make_refs_multi(vps, generation_config, seed=42, M=globals().get("_MR_M",1), content=globals().get("_MR_CONTENT",FIXED_CONTENT))
            for i in range(0, len(texts), STAGE2_BATCH):
                outs = stage2_continuation_batch(texts[i:i+STAGE2_BATCH], refs[i:i+STAGE2_BATCH], generation_config)
                for out in outs:
                    wavs, sample_rate = continuation_decode(out)
                    audio_values.append(wavs[0])

        # Save audio
        for pth, adv in zip(paths, audio_values):
            sf.write(pth, adv, sample_rate)
            try:
                pth.stat().st_size > 0
            except FileNotFoundError:
                return False

        random.setstate(rng_state['random'])
        np.random.set_state(rng_state['numpy'])
        torch.set_rng_state(rng_state['torch'])
        if rng_state['cuda']:
            torch.cuda.set_rng_state_all(rng_state['cuda'])

        import gc
        gc.collect()
        torch.cuda.empty_cache()
        return True


In [17]:
from enum import Enum

class ModelType(Enum):
    TEST = model.__class__.__name__


test_model_type = ModelType.TEST
test_model = TestModel()

In [18]:
from pathlib import Path

def save_and_evaluate(model, output_dir: str, disable_evaluate: bool = False, disable_save: bool = False):
    evaluator = EvaluationPipeline(base_dir=Path(output_dir), html=True, verbose=False)
    test_config.generation.output_dir = Path(output_dir)

    model.eval()
    if not disable_save:
        model.save_pretrained(output_dir)

    if not disable_evaluate:
        strategy = create_strategy(GenerationMethod.METHOD2, test_config, test_dataset, test_model)
        exp2_result = strategy.generate_batch_group_all(test_dataset_type.value, test_model_type.value)

        exp2_eval_result = evaluator.evaluate_dataset_model(
            dataset_type=test_dataset_type,
            model_type=test_model_type,
            methods=[GenerationMethod.METHOD2]
        )
        #evaluator.save_results_to_csv(exp2_eval_result, test_dataset_type, test_model_type)
        return exp2_eval_result


## Run Gen-Gen Evaluation

기존 `save_and_evaluate` 파이프라인을 그대로 호출. 학습이 없으므로 `disable_evaluate=False`만 지정한다.

In [19]:
import json as _json
MED = FIXED_CONTENT   # ~medium sentence (fixedanchor default)
SHORT = "Hello, this is a short reference."
CONFIGS = [
  ("mr_M1_long",  1, MED),
  ("mr_M4_long",  4, MED),
  ("mr_M8_short", 8, SHORT),
]
os.makedirs("results/_summary", exist_ok=True)
mr_results = {}
for _name, _M, _c in CONFIGS:
    globals()["_MR_M"] = _M; globals()["_MR_CONTENT"] = _c
    OUTPUT_DIR = f"./results/{model_id}_{_name}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"\n===== {_name}  M={_M}  content={_c!r} =====", flush=True)
    _res = save_and_evaluate(model, OUTPUT_DIR, disable_save=True)
    _m = list(_res.values())[0]
    mr_results[_name] = {"M": _M, **{k: float(_m[k]) for k in ["sim_mean","utmos_mean","wer_mean"]}}
    _json.dump({"name": _name, "M": _M, "content": _c, "metrics": mr_results[_name]}, open(f"results/_summary/{_name}.json","w"), indent=2)
    print(f"[{_name}] COS={_m['sim_mean']:.4f} UTMOS={_m['utmos_mean']:.3f} WER={_m['wer_mean']:.3f}", flush=True)
print("\n==== MULTI-REFERENCE ====")
for _n,_v in mr_results.items():
    print(f"  {_n:12s} M={_v['M']}  COS={_v['sim_mean']:.4f}  WER={_v['wer_mean']:.3f}")
mr_results


===== mr_M1_long  M=1  content='Hello, this is a fixed sentence used as the acoustic anchor for voice design retrieval.' =====


Processing references:   0%|          | 0/10 [00:00<?, ?it/s]

Set 0:   0%|          | 0/10 [00:00<?, ?it/s]

Set 1:   0%|          | 0/10 [00:00<?, ?it/s]

Set 2:   0%|          | 0/10 [00:00<?, ?it/s]

...

Error calculating mcd: Failed to create calculator for MetricType.MCD. Available metrics: [<MetricType.UTMOS: 'utmos'>, <MetricType.WER: 'wer'>, <MetricType.SIM: 'sim'>, <MetricType.FFE: 'ffe'>, <MetricType.MCD: 'mcd'>]


Metric,Mean,Std,Median,Avg Std,Avg CV
UTMOS,3.3904,0.5188,3.5166,0.3381,0.1013
WER,0.1558,0.1178,0.1714,0.1122,0.7314
COS,0.3482,0.3299,0.2748,0.0634,0.7658
FFE,0.5393,0.1083,0.5625,0.0592,0.1178


[mr_M8_short] COS=0.3482 UTMOS=3.390 WER=0.156



==== MULTI-REFERENCE ====
  mr_M1_long   M=1  COS=0.3938  WER=0.124
  mr_M4_long   M=4  COS=0.3452  WER=0.121
  mr_M8_short  M=8  COS=0.3482  WER=0.156


{'mr_M1_long': {'M': 1,
  'sim_mean': 0.39377478466679655,
  'utmos_mean': 3.3423828125,
  'wer_mean': 0.1241394943076009},
 'mr_M4_long': {'M': 4,
  'sim_mean': 0.34522655110599265,
  'utmos_mean': 3.439853515625,
  'wer_mean': 0.12106447410864113},
 'mr_M8_short': {'M': 8,
  'sim_mean': 0.3481758563158413,
  'utmos_mean': 3.3903515625,
  'wer_mean': 0.15576612415517246}}